In [1]:
# Objective : Implement Historical Value at Risk with GARCH Model

In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
from arch import arch_model

In [3]:
#1 Fetch the data
data = yf.download(tickers=["GS", "MS", "JPM"], start = "2020-01-01", end = "2026-08-10", auto_adjust=True)
data = data['Close']
data.head()

[*********************100%***********************]  3 of 3 completed


Ticker,GS,JPM,MS
Date,,,
2020-01-02,200.251297,117.313927,42.114906
2020-01-03,197.909622,115.765793,41.435116
2020-01-06,199.935089,115.673775,41.289440
2020-01-07,201.251190,113.707253,41.208519
2020-01-08,203.191116,114.594261,41.734547


In [4]:
#2 Calculate individual and portfolio return
returns = data.pct_change() # (St - St-1)/St-1
returns = returns.dropna()
returns

Ticker,GS,JPM,MS
Date,,,
2020-01-03,-0.011694,-0.013197,-0.016141
2020-01-06,0.010234,-0.000795,-0.003516
2020-01-07,0.006583,-0.017001,-0.001960
2020-01-08,0.009639,0.007801,0.012765
2020-01-09,0.020357,0.003652,0.009501
...,...,...,...
2026-08-03,0.008523,0.002416,0.003849
2026-08-04,0.025237,0.013838,0.027506
2026-08-05,0.007028,0.004811,0.005667


In [5]:
#3 Calculate Portfolio Return

portfolio_value = 1000000 #1M
weights = np.array([0.4, 0.3, 0.3])
portfolio_returns = returns.dot(weights)
portfolio_returns = portfolio_returns*100
portfolio_returns

Date
2020-01-03   -1.347882
2020-01-06    0.280054
2020-01-07   -0.305508
2020-01-08    1.002548
2020-01-09    1.208857
                ...   
2026-08-03    0.528905
2026-08-04    2.249800
2026-08-05    0.595449
2026-08-06   -1.915451
2026-08-07    0.737162
Length: 1657, dtype: float64

In [6]:
#4 Fit the GARCH Model

garch_model = arch_model(portfolio_returns, vol='Garch', p=1, q=1, dist="normal") #Calling the model
results = garch_model.fit(disp='off') #Fitting the GARCH model to the data
print(results.summary())

                     Constant Mean - GARCH Model Results                      
Dep. Variable:                   None   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -3197.97
Distribution:                  Normal   AIC:                           6403.94
Method:            Maximum Likelihood   BIC:                           6425.59
                                        No. Observations:                 1657
Date:                Tue, Nov 10 2026   Df Residuals:                     1656
Time:                        15:50:50   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu             0.1472  3.823e-02      3.850  1.183e-04 [7.225e-0

In [7]:
#4 Standardized residuals
standardized_shock = results.std_resid
standardized_shock

# Actual returns = expected returns + epsilon
# r_t = mu + epsilon
# epsilon = r_t - mu ...(1) #Shock Parameter
# epsilon = z*sigma ...(2)
# Compare 1 and 2
# r_t - mu = z*sigma
# z = (r_t - mu)/sigma #Sigma -> Conditional Volatility that we get from GARCH  -> Standardizing the shock

Date
2020-01-03   -0.608799
2020-01-06    0.058366
2020-01-07   -0.221304
2020-01-08    0.459498
2020-01-09    0.611758
                ...   
2026-08-03    0.174707
2026-08-04    1.064938
2026-08-05    0.225847
2026-08-06   -1.138712
2026-08-07    0.317677
Name: std_resid, Length: 1657, dtype: float64

In [8]:
#5 Forecasting using GARCH Model
forecast = results.forecast(horizon = 1, reindex= False)
tomorrow_variance = forecast.variance.iloc[-1,0] # Variance
tomorrow_volatility = np.sqrt(tomorrow_variance)
print(tomorrow_volatility)

1.7137676083775466


In [9]:
#6 Rescale the historical shock using GARCH Volatility
# actual_return = mu + epsilon
# actual_return(t+1) = mu + sigma(t+1)*z => Scenario shocks

mean_return = results.params[0] #mu
scenario_returns_pct = mean_return + tomorrow_volatility * standardized_shock
scenario_returns_decimal = scenario_returns_pct/100
scenario_returns_decimal  # Shocks which are scaled to volatility


KeyError: 0

In [ ]:
#7 Confidence Interval
confidence_interval = [0.90, 0.95, 0.99]
for confidence in confidence_interval:
  alpha = 1 - confidence
  VaR_returns = np.percentile(scenario_returns_decimal, alpha*100)
  VaR_returns = -VaR_returns
  VaR_dollar = VaR_returns*portfolio_value

  print("Historical Value at Risk + GARCH", "@", confidence*100, "%: ", "$", VaR_dollar)

Historical Value at Risk + GARCH @ 90.0 %:  $ 18965.219888161402
Historical Value at Risk + GARCH @ 95.0 %:  $ 25639.891420624797
Historical Value at Risk + GARCH @ 99.0 %:  $ 41972.31963533468


## Objective : Implement Historical Value at Risk + GARCH (student t distribution)

In [ ]:
#1 Fetch the data
data = yf.download(tickers=["GS", "MS", "JPM"], start = "2020-01-01", end = "2026-08-10", auto_adjust=True)
data = data['Close']
data.head()

[*********************100%***********************]  3 of 3 completed


Ticker,GS,JPM,MS
Date,,,
2020-01-02,200.251312,117.313927,42.371357
2020-01-03,197.909668,115.765808,41.687428
2020-01-06,199.935074,115.673759,41.540878
2020-01-07,201.251175,113.707230,41.459454
2020-01-08,203.191116,114.594284,41.988689


In [ ]:
#2 Calculate individual and portfolio return
returns = data.pct_change() # (St - St-1)/St-1
returns = returns.dropna()
returns

Ticker,GS,JPM,MS
Date,,,
2020-01-03,-0.011694,-0.013196,-0.016141
2020-01-06,0.010234,-0.000795,-0.003515
2020-01-07,0.006583,-0.017001,-0.001960
2020-01-08,0.009639,0.007801,0.012765
2020-01-09,0.020357,0.003651,0.009502
...,...,...,...
2026-08-03,0.008523,0.002416,0.003849
2026-08-04,0.025237,0.013838,0.027506
2026-08-05,0.007028,0.004811,0.005667


In [ ]:
#3 Calculate Portfolio Return

portfolio_value = 1000000 #1M
weights = np.array([0.4, 0.3, 0.3])
portfolio_returns = returns.dot(weights)
portfolio_returns = portfolio_returns*100
portfolio_returns

,0
Date,
2020-01-03,-1.347872
2020-01-06,0.280043
2020-01-07,-0.305517
2020-01-08,1.002567
2020-01-09,1.208851
...,...
2026-08-03,0.528904
2026-08-04,2.249800
2026-08-05,0.595449


In [ ]:
#4 Fit the GARCH Model

garch_model = arch_model(portfolio_returns, vol='Garch', p=1, q=1, dist="t") #Calling the model
results = garch_model.fit(disp='off') #Fitting the GARCH model to the data
print(results.summary())

                        Constant Mean - GARCH Model Results                         
Dep. Variable:                         None   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                            GARCH   Log-Likelihood:               -3140.76
Distribution:      Standardized Student's t   AIC:                           6291.52
Method:                  Maximum Likelihood   BIC:                           6318.58
                                              No. Observations:                 1657
Date:                      Sun, Oct 25 2026   Df Residuals:                     1656
Time:                              12:18:35   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu        

In [ ]:
#4 Standardized residuals
standardized_shock = results.std_resid
standardized_shock

# Actual returns = expected returns + epsilon
# r_t = mu + epsilon
# epsilon = r_t - mu ...(1) #Shock Parameter
# epsilon = z*sigma ...(2)
# Compare 1 and 2
# r_t - mu = z*sigma
# z = (r_t - mu)/sigma #Sigma -> Conditional Volatility that we get from GARCH  -> Standardizing the shock

,std_resid
Date,
2020-01-03,-0.603572
2020-01-06,0.062159
2020-01-07,-0.207793
2020-01-08,0.439885
2020-01-09,0.578761
...,...
2026-08-03,0.184310
2026-08-04,1.071930
2026-08-05,0.232325


In [ ]:
#5 Forecasting using GARCH Model
forecast = results.forecast(horizon = 1, reindex= False)
tomorrow_variance = forecast.variance.iloc[-1,0] # Variance
tomorrow_volatility = np.sqrt(tomorrow_variance)
print(tomorrow_volatility)

1.7503428384263164


In [ ]:
#6 Rescale the historical shock using GARCH Volatility
# actual_return = mu + epsilon
# actual_return(t+1) = mu + sigma(t+1)*z => Scenario shocks

mean_return = results.params[0] #mu
scenario_returns_pct = mean_return + tomorrow_volatility * standardized_shock
scenario_returns_decimal = scenario_returns_pct/100
scenario_returns_decimal  # Shocks which are scaled to volatility


/tmp/ipykernel_3038/1397668756.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  mean_return = results.params[0] #mu


,std_resid
Date,
2020-01-03,-0.009202
2020-01-06,0.002451
2020-01-07,-0.002274
2020-01-08,0.009062
2020-01-09,0.011493
...,...
2026-08-03,0.004589
2026-08-04,0.020125
2026-08-05,0.005429


In [ ]:
#7 Confidence Interval
confidence_interval = [0.90, 0.95, 0.99]
for confidence in confidence_interval:
  alpha = 1 - confidence
  VaR_returns = np.percentile(scenario_returns_decimal, alpha*100)
  VaR_returns = -VaR_returns
  VaR_dollar = VaR_returns*portfolio_value

  print("Historical Value at Risk + GARCH", "@", confidence*100, "%: ", "$", VaR_dollar)

Historical Value at Risk + GARCH @ 90.0 %:  $ 19653.64811050811
Historical Value at Risk + GARCH @ 95.0 %:  $ 26314.19779802163
Historical Value at Risk + GARCH @ 99.0 %:  $ 43442.717631884756


In [ ]:
# How do you build scenario for HVaR

# Historical_Scenarios = mu + sigma*z
# mu -> GARCH Model
# sigma -> tomorrow vol -> GARCH Model
# z -> shock => (R_T - mu)/sigma

# Historical Value at Risk + GJR GARCH

In [ ]:
#1 Fetch the data
data = yf.download(tickers=["GS", "MS", "JPM"], start = "2020-01-01", end = "2026-08-10", auto_adjust=True)
data = data['Close']
data.head()

[*********************100%***********************]  3 of 3 completed


Ticker,GS,JPM,MS
Date,,,
2020-01-02,200.251312,117.313927,42.371357
2020-01-03,197.909668,115.765808,41.687428
2020-01-06,199.935074,115.673759,41.540878
2020-01-07,201.251175,113.707230,41.459454
2020-01-08,203.191116,114.594284,41.988689


In [ ]:
#2 Calculate individual and portfolio return
returns = data.pct_change() # (St - St-1)/St-1
returns = returns.dropna()
returns

Ticker,GS,JPM,MS
Date,,,
2020-01-03,-0.011694,-0.013196,-0.016141
2020-01-06,0.010234,-0.000795,-0.003515
2020-01-07,0.006583,-0.017001,-0.001960
2020-01-08,0.009639,0.007801,0.012765
2020-01-09,0.020357,0.003651,0.009502
...,...,...,...
2026-08-03,0.008523,0.002416,0.003849
2026-08-04,0.025237,0.013838,0.027506
2026-08-05,0.007028,0.004811,0.005667


In [ ]:
#3 Calculate Portfolio Return

portfolio_value = 1000000 #1M
weights = np.array([0.4, 0.3, 0.3])
portfolio_returns = returns.dot(weights) # w1r1 + w2r2 + w3r3
portfolio_returns = portfolio_returns*100
portfolio_returns

,0
Date,
2020-01-03,-1.347872
2020-01-06,0.280043
2020-01-07,-0.305517
2020-01-08,1.002567
2020-01-09,1.208851
...,...
2026-08-03,0.528904
2026-08-04,2.249800
2026-08-05,0.595449


In [ ]:
#4 Fit the GJR GARCH Model with student t distribution

garch_model = arch_model(portfolio_returns, vol='Garch', p=1,o=1, q=1, dist="t") #Calling the model
results = garch_model.fit(disp='off') #Fitting the GARCH model to the data
print(results.summary())

                      Constant Mean - GJR-GARCH Model Results                       
Dep. Variable:                         None   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                        GJR-GARCH   Log-Likelihood:               -3123.85
Distribution:      Standardized Student's t   AIC:                           6259.70
Method:                  Maximum Likelihood   BIC:                           6292.18
                                              No. Observations:                 1657
Date:                      Sun, Oct 25 2026   Df Residuals:                     1656
Time:                              12:19:44   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu        

In [ ]:
#5 Standardized residuals
standardized_shock = results.std_resid
standardized_shock

# Actual returns = expected returns + epsilon
# r_t = mu + epsilon
# epsilon = r_t - mu ...(1) #Shock Parameter
# epsilon = z*sigma ...(2)
# Compare 1 and 2
# r_t - mu = z*sigma
# z = (r_t - mu)/sigma #Sigma -> Conditional Volatility that we get from GARCH  -> Standardizing the shock

,std_resid
Date,
2020-01-03,-0.587339
2020-01-06,0.071838
2020-01-07,-0.182377
2020-01-08,0.418546
2020-01-09,0.544632
...,...
2026-08-03,0.191239
2026-08-04,1.029903
2026-08-05,0.246179


In [ ]:
#6 Forecasting using GJR GARCH Model
forecast = results.forecast(horizon = 1, reindex= False)
tomorrow_variance = forecast.variance.iloc[-1,0] # Variance
tomorrow_volatility = np.sqrt(tomorrow_variance)
print(tomorrow_volatility)

1.873053282893983


In [ ]:
#7 Rescale the historical shock using GARCH Volatility
# actual_return = mu + epsilon
# actual_return(t+1) = mu + sigma(t+1)*z => Scenario shocks

mean_return = results.params[0] #mu
scenario_returns_pct = mean_return + tomorrow_volatility * standardized_shock
scenario_returns_decimal = scenario_returns_pct/100
scenario_returns_decimal  # Shocks which are scaled to volatility


/tmp/ipykernel_3038/3214553726.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  mean_return = results.params[0] #mu


,std_resid
Date,
2020-01-03,-0.009929
2020-01-06,0.002417
2020-01-07,-0.002344
2020-01-08,0.008911
2020-01-09,0.011273
...,...
2026-08-03,0.004654
2026-08-04,0.020362
2026-08-05,0.005683


In [ ]:
#7 Confidence Interval
confidence_interval = [0.90, 0.95, 0.99]
for confidence in confidence_interval:
  alpha = 1 - confidence
  VaR_returns = np.percentile(scenario_returns_decimal, alpha*100)
  VaR_returns = -VaR_returns
  VaR_dollar = VaR_returns*portfolio_value

  print("Historical Value at Risk + GJR GARCH", "@", confidence*100, "%: ", "$", VaR_dollar)

Historical Value at Risk + GJR GARCH @ 90.0 %:  $ 21503.60360089589
Historical Value at Risk + GJR GARCH @ 95.0 %:  $ 29173.070967933858
Historical Value at Risk + GJR GARCH @ 99.0 %:  $ 45617.39722447335


# Historical Value at Risk + EGARCH Model

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from arch import arch_model

In [ ]:
#1. Download the data from yahoo finance
data = yf.download(tickers = ["GS", "MS", "JPM"], start ="2020-01-01", end = "2026-10-08", auto_adjust = True)
data = data['Close']
data

[*********************100%***********************]  3 of 3 completed


Ticker,GS,JPM,MS
Date,,,
2020-01-02,200.251266,117.313934,42.371380
2020-01-03,197.909668,115.765808,41.687428
2020-01-06,199.935089,115.673729,41.540874
2020-01-07,201.251190,113.707230,41.459457
2020-01-08,203.191132,114.594276,41.988693
...,...,...,...
2026-10-01,896.669983,331.526031,188.009995
2026-10-02,902.559998,330.730011,190.309998
2026-10-05,893.460022,330.730011,190.190002


In [ ]:
#2 Calculate individual return
returns = data.pct_change().dropna()
weights = np.array([0.40, 0.30, 0.30])
portfolio_value = 1000000

portfolio_returns = returns.dot(weights)
portfolio_returns = portfolio_returns*100
portfolio_returns

,0
Date,
2020-01-03,-1.347881
2020-01-06,0.280035
2020-01-07,-0.305504
2020-01-08,1.002565
2020-01-09,1.208839
...,...
2026-10-01,0.038001
2026-10-02,0.557720
2026-10-05,-0.422212


In [ ]:
#3 Fit EGARCH(1,1) model on portfolio returns
egarch_model = arch_model(
    portfolio_returns,
    mean = "Constant",
    vol = "EGARCH",
    p = 1,
    q = 1,
    dist = "t")

egarch_result = egarch_model.fit(disp = "off")
print(egarch_result.summary())

                        Constant Mean - EGARCH Model Results                        
Dep. Variable:                         None   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                           EGARCH   Log-Likelihood:               -3210.12
Distribution:      Standardized Student's t   AIC:                           6430.24
Method:                  Maximum Likelihood   BIC:                           6457.43
                                              No. Observations:                 1699
Date:                      Sun, Oct 25 2026   Df Residuals:                     1698
Time:                              12:19:49   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu        

In [ ]:
#4 Extract the standardized residuals (z_t)
standardized_shocks = egarch_result.std_resid
standardized_shocks

# actual_returns = mean_return + error
# error = sigma * z
# z = error/sigma


,std_resid
Date,
2020-01-03,-0.595077
2020-01-06,0.064820
2020-01-07,-0.196020
2020-01-08,0.428632
2020-01-09,0.552194
...,...
2026-10-01,-0.051237
2026-10-02,0.275037
2026-10-05,-0.364287


In [ ]:
#5 Tomorrow's volatility
forecast = egarch_result.forecast(horizon = 1, reindex = False)
tomorrow_variance = forecast.residual_variance.iloc[-1, 0]
tomorrow_volatility = np.sqrt(tomorrow_variance)
tomorrow_volatility

np.float64(1.371099010186607)

In [ ]:
# 6. Rescale Historical Shocks to Tomorrow's Volatility Regime
mean_return = egarch_result.params["mu"]
scenario_returns = mean_return + tomorrow_volatility * standardized_shocks
scenario_returns = scenario_returns/100
scenario_returns

# Historical scenarios but are scaled to volatility (EGARCH Model)

,std_resid
Date,
2020-01-03,-0.006911
2020-01-06,0.002137
2020-01-07,-0.001440
2020-01-08,0.007125
2020-01-09,0.008819
...,...
2026-10-01,0.000546
2026-10-02,0.005019
2026-10-05,-0.003747


In [ ]:
#7 Calculate VaR at difference confidence interval

confidence_levels = [0.90, 0.95, 0.99]
for conf in confidence_levels:
  alpha = 1 - conf
  var_percentile = np.percentile(scenario_returns, alpha*100)
  var_percentile = - var_percentile
  var_dollar = var_percentile * portfolio_value

  print(f"Confidence Level: {conf*100:.0f}%")
  print(f"  Dollar VaR ($100k):    ${var_dollar:,.2f}\n")
  print("="*50)

Confidence Level: 90%
  Dollar VaR ($100k):    $15,168.06

Confidence Level: 95%
  Dollar VaR ($100k):    $21,152.59

Confidence Level: 99%
  Dollar VaR ($100k):    $34,042.77



# Historical VaR + EWMA

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf

In [ ]:
#1 Download the data from Yahoo Finance

data = yf.download(tickers=["GS", "MS", "JPM"], start = "2020-01-01", end = "2026-08-10", auto_adjust=True)
data = data['Close']
data

[*********************100%***********************]  3 of 3 completed


Ticker,GS,JPM,MS
Date,,,
2020-01-02,200.251312,117.313927,42.371357
2020-01-03,197.909668,115.765808,41.687428
2020-01-06,199.935074,115.673759,41.540878
2020-01-07,201.251175,113.707230,41.459454
2020-01-08,203.191116,114.594284,41.988689
...,...,...,...
2026-08-03,1022.054443,350.889435,211.229996
2026-08-04,1047.848022,355.745209,217.039993
2026-08-05,1055.211914,357.456665,218.270004


In [ ]:
#2 Calculate Portfolio Returns
returns = data.pct_change().dropna()
portffolio_value = 1000000
weights = np.array([0.4, 0.3, 0.3])
portfolio_returns = returns.dot(weights) # Rp = w1r1 + w2r2 + w3r3
portfolio_returns = portfolio_returns * 100
portfolio_returns

,0
Date,
2020-01-03,-1.347872
2020-01-06,0.280043
2020-01-07,-0.305517
2020-01-08,1.002567
2020-01-09,1.208851
...,...
2026-08-03,0.528904
2026-08-04,2.249800
2026-08-05,0.595449


In [ ]:
#3 Initialize EWMA Variance
#EWMA sigma^2(t) = lamda * sigma^2(t-1) + (1-lamda)*return^2
decay_factor = 0.94 #Effect of shocks on volatility should be for shorter duration (~11 days)

first_30day_returns = portfolio_returns.iloc[:30]
variance = np.mean(first_30day_returns**2)  #Variance = (1 / n-1) *(sum ((X - Xavg))
volatility = np.sqrt(variance)
volatility

# Avg of Daily Returns = 0

np.float64(1.2056557025540362)

In [ ]:
#4 Calculate standard shocks and update the variance

# Standardized first 30 day return
standardized_shocks = list(first_30day_returns / volatility)
standardized_shocks

[-1.1179574994106187,
 0.23227437807280582,
 -0.2534034527836256,
 0.8315532088227398,
 1.00265039224373,
 -0.2433472207819169,
 0.9031022765011995,
 0.3887705937145931,
 -0.3959972344354853,
 2.351728988361417,
 0.6094141338137117,
 -1.4328659373024701,
 0.12648946418198337,
 -0.36988908922249963,
 -1.551370130880522,
 -1.3845500823217154,
 1.4972319846554305,
 -0.530066063569934,
 1.0903911304301608,
 -2.2346588359816932,
 0.5275937044130619,
 1.436665690329694,
 1.1934426905135311,
 -0.3241270601535965,
 -0.5946020478022079,
 0.15887370964948702,
 0.11448258436364858,
 0.48570397690699035,
 -0.174073874274925,
 -0.29261214880675596]

In [ ]:
#5 Forecasted tomorrow's Volatility
remaining_returns = portfolio_returns.iloc[30:]

for daily_returns in remaining_returns:
  volatility = np.sqrt(variance)
  shocks = daily_returns / volatility
  standardized_shocks.append(shocks)

  # Update the variance for next trading day?
  variance = decay_factor * variance + (1 - decay_factor) * daily_returns**2

tomorrow_volatility = np.sqrt(variance)
tomorrow_volatility

np.float64(1.931041599801792)

In [ ]:
#6 Rescale Historical shocks using tomorrow's volatility
standardized_shocks = np.array(standardized_shocks)
scenario_returns_pct = tomorrow_volatility * standardized_shocks
scenario_returns_decimal = scenario_returns_pct/100
scenario_returns_decimal

array([-0.02158822,  0.00448531, -0.00489333, ...,  0.00562505,
       -0.01861298,  0.00717849])

In [ ]:
#7 Calculate VaR

confidence_interval = [0.90, 0.95, 0.99]

for conf in confidence_interval:
  alpha = 1 - conf
  var_percentile = np.percentile(scenario_returns_decimal, alpha*100)
  var_percentile = - var_percentile
  var_dollar = var_percentile * portfolio_value
  print("Historical Value at Risk @", conf*100, "%" ," interval is : $", var_dollar)

Historical Value at Risk @ 90.0 %  interval is : $ 22584.40586495472
Historical Value at Risk @ 95.0 %  interval is : $ 32449.63528006419
Historical Value at Risk @ 99.0 %  interval is : $ 52171.27012783141
